# Comparación Final de Representaciones: BoW vs Word2Vec vs BERT


> Carga los embeddings ya guardados en MongoDB (Word2Vec y BERT) y los compara
> contra TF-IDF mediante clasificación, clustering y visualización t-SNE.
> Las canciones están en **inglés** → se usó `bert-base-uncased`.


## Imports y configuración

In [ ]:
import warnings, os, re
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
import sys
sys.path.append(os.path.abspath('../../../../../..'))
from src.data.mongo_storage import _get_default_collection

FIGS = Path('../../../../../..') / 'data' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.facecolor':'white','axes.grid':True,'grid.alpha':0.3,'font.size':11})
PALETTE = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860','#DA8BC3','#8C8C8C','#CCB974','#64B5CD']
print('✓ Imports OK')


## Cargar canciones con embeddings desde MongoDB

In [ ]:
col = _get_default_collection()

# Solo canciones que tienen AMBOS embeddings guardados
docs = list(col.find(
    {
        'embeddings.word2vec_avg': {'$exists': True, '$ne': [], '$not': {'$size': 0}},
        'embeddings.beto_cls':     {'$exists': True, '$ne': [], '$not': {'$size': 0}},
        'Lyrics': {'$ne': None},
        'Genre':  {'$ne': None},
    },
    {'_id': 1, 'Song': 1, 'Artist': 1, 'Genre': 1,
     'Song year': 1, 'Lyrics': 1, 'embeddings': 1}
))

df = pd.DataFrame(docs)
df['Lyrics'] = df['Lyrics'].astype(str)

# Extraer embeddings a columnas separadas
df['w2v_emb']  = df['embeddings'].apply(lambda x: x.get('word2vec_avg', []))
df['beto_emb'] = df['embeddings'].apply(lambda x: x.get('beto_cls', []))

# Filtrar filas con embeddings válidos (no vacíos)
df = df[df['w2v_emb'].apply(len) > 0].copy()
df = df[df['beto_emb'].apply(len) > 0].copy()
df = df.reset_index(drop=True)

# Solo géneros con >= 10 canciones en la muestra
generos_validos = df['Genre'].value_counts()
generos_validos = generos_validos[generos_validos >= 10].index.tolist()
df = df[df['Genre'].isin(generos_validos)].reset_index(drop=True)

print(f'✓ {len(df):,} canciones con embeddings completos | {df["Genre"].nunique()} géneros')
print(df['Genre'].value_counts().to_string())


## Construir matrices de representación

In [ ]:
# 1. TF-IDF (BoW) — generado aquí, no almacenado en Mongo
print('Generando TF-IDF...')
tfidf_vec = TfidfVectorizer(max_features=5000, stop_words='english', min_df=2)
X_tfidf = tfidf_vec.fit_transform(df['Lyrics'].tolist()).toarray()
print(f'  TF-IDF shape: {X_tfidf.shape}')

# 2. Word2Vec (ya en MongoDB)
X_w2v = np.array(df['w2v_emb'].tolist())
print(f'  Word2Vec shape: {X_w2v.shape}')

# 3. BERT bert-base-uncased (ya en MongoDB como beto_cls)
X_bert = np.array(df['beto_emb'].tolist())
print(f'  BERT shape: {X_bert.shape}')

# Etiquetas
le = LabelEncoder()
LABELS = le.fit_transform(df['Genre'].tolist())

REPRESENTACIONES = {
    'TF-IDF (BoW)': X_tfidf,
    'Word2Vec':     X_w2v,
    'BERT [CLS]':   X_bert,
}
print('\n✓ Matrices listas')


## Clasificación de género — Regresión Logística

In [ ]:
print('=== Clasificación de Género (Logistic Regression, 75/25 split) ===\n')
clf_acc     = {}
clf_reports = {}

for nombre, X in REPRESENTACIONES.items():
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_sc, LABELS, test_size=0.25, random_state=42, stratify=LABELS
    )
    clf = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
    clf.fit(X_tr, y_tr)
    acc = clf.score(X_te, y_te)
    clf_acc[nombre]     = acc
    clf_reports[nombre] = classification_report(
        y_te, clf.predict(X_te), target_names=le.classes_, output_dict=True
    )
    print(f'  {nombre:<20s}: accuracy = {acc:.4f}  ({acc*100:.1f}%)')

mejor_clf = max(clf_acc, key=clf_acc.get)
print(f'\n🏆 Mejor: {mejor_clf} ({clf_acc[mejor_clf]:.4f})')


## Clustering K-Means + Silhouette Score

In [ ]:
n_clusters = len(generos_validos)
print(f'=== Clustering K-Means (k={n_clusters}) ===\n')
sil_scores = {}

for nombre, X in REPRESENTACIONES.items():
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = km.fit_predict(X_sc)
    sil = silhouette_score(X_sc, cluster_labels)
    sil_scores[nombre] = sil
    print(f'  {nombre:<20s}: Silhouette = {sil:.4f}')

mejor_sil = max(sil_scores, key=sil_scores.get)
print(f'\n🏆 Mejor clustering: {mejor_sil} ({sil_scores[mejor_sil]:.4f})')
print('  (1.0 = perfecto | 0 = solapados | negativo = mal asignados)')


## Gráfico comparativo — Clasificación y Clustering

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
nombres = list(clf_acc.keys())
colores = PALETTE[:len(nombres)]

# Clasificación
accs = [clf_acc[n] for n in nombres]
bars = axes[0].bar(nombres, accs, color=colores, edgecolor='white', linewidth=1.5)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Clasificación de Género\n(Logistic Regression, 75/25 split)', fontweight='bold')
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
                 f'{acc:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Clustering
sils = [sil_scores[n] for n in nombres]
bars2 = axes[1].bar(nombres, sils, color=colores, edgecolor='white', linewidth=1.5)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title(f'Calidad de Clustering\n(K-Means, k={n_clusters})', fontweight='bold')
for bar, sil in zip(bars2, sils):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{sil:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.suptitle('Comparación de Representaciones: BoW vs Word2Vec vs BETO',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGS / 'comparacion_representaciones.png', dpi=130, bbox_inches='tight')
plt.show()


## Visualización t-SNE — Las tres representaciones comparadas

El doc requiere proyecciones 2D de **cada representación** para comparar
visualmente la separación entre géneros.


In [ ]:
print('Calculando t-SNE para las 3 representaciones...')
print('(puede tardar varios minutos en CPU)')

def calcular_tsne(X: np.ndarray, nombre: str) -> np.ndarray:
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)
    # PCA primero: acelera t-SNE y lo estabiliza
    n_comp_pca = min(50, X_sc.shape[1])
    pca = PCA(n_components=n_comp_pca, random_state=42)
    X_pca = pca.fit_transform(X_sc)
    print(f'  {nombre}: PCA {X_sc.shape[1]}d → {n_comp_pca}d', end=' ... ')
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
    X_2d = tsne.fit_transform(X_pca)
    print('OK')
    return X_2d

tsne_resultados = {nombre: calcular_tsne(X, nombre)
                   for nombre, X in REPRESENTACIONES.items()}

# ── Plot: 1 fila × 3 columnas ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (nombre, X_2d) in zip(axes, tsne_resultados.items()):
    for i, genero in enumerate(generos_validos):
        mask = df['Genre'] == genero
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   label=genero, alpha=0.55, s=18,
                   color=PALETTE[i % len(PALETTE)])
    ax.set_title(nombre, fontsize=12, fontweight='bold')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.grid(True, alpha=0.3)

# Leyenda compartida fuera del último subplot
handles, labels_leg = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center',
           ncol=len(generos_validos), fontsize=9,
           bbox_to_anchor=(0.5, -0.08))

plt.suptitle('t-SNE: Separación entre géneros por representación',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGS / 'tsne_comparacion_representaciones.png',
            dpi=130, bbox_inches='tight')
plt.show()

# También guardar t-SNE individual de BERT para referencia
fig2, ax2 = plt.subplots(figsize=(10, 7))
X_2d_bert = tsne_resultados['BERT [CLS]']
for i, genero in enumerate(generos_validos):
    mask = df['Genre'] == genero
    ax2.scatter(X_2d_bert[mask, 0], X_2d_bert[mask, 1],
                label=genero, alpha=0.6, s=25,
                color=PALETTE[i % len(PALETTE)])
ax2.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax2.set_title('t-SNE — BERT [CLS] por Género Musical',
              fontsize=13, fontweight='bold')
ax2.set_xlabel('t-SNE 1'); ax2.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig(FIGS / 'tsne_bert_generos.png', dpi=130, bbox_inches='tight')
plt.show()


## Reporte detallado por género (mejor representación)

In [ ]:
print(f'=== Classification Report — {mejor_clf} ===\n')
report = clf_reports[mejor_clf]
rows = []
for genero in le.classes_:
    r = report.get(genero, {})
    rows.append({
        'Género':    genero,
        'Precision': round(r.get('precision', 0), 3),
        'Recall':    round(r.get('recall', 0), 3),
        'F1-Score':  round(r.get('f1-score', 0), 3),
        'Support':   int(r.get('support', 0)),
    })
df_report = pd.DataFrame(rows).sort_values('F1-Score', ascending=False)
print(df_report.to_string(index=False))


## Resumen final

In [ ]:
print('=' * 65)
print('  RESUMEN — Comparación de Representaciones Vectoriales')
print('=' * 65)
print(f'  Corpus: {len(df):,} canciones | {len(generos_validos)} géneros')
print()
print('  CLASIFICACIÓN (Logistic Regression):')
for n, acc in sorted(clf_acc.items(), key=lambda x: -x[1]):
    star = '🏆' if n == mejor_clf else '  '
    print(f'  {star} {n:<22s} acc = {acc:.4f}')
print()
print('  CLUSTERING (Silhouette Score):')
for n, sil in sorted(sil_scores.items(), key=lambda x: -x[1]):
    star = '🏆' if n == mejor_sil else '  '
    print(f'  {star} {n:<22s} sil = {sil:.4f}')
print()
print('  t-SNE (separación visual entre géneros):')
print('  • TF-IDF : clusters compactos pero dispersos en dimensión original')
print('  • Word2Vec: agrupaciones semánticas visibles')
print('  • BERT   : mejor separación contextual entre géneros similares')
print()
print('  CONCLUSIONES CLAVE:')
print('  • TF-IDF: alto rendimiento en clasificación, ignora semántica')
print('  • Word2Vec: captura relaciones semánticas (analogías, vecinos)')
print('  • BERT: embeddings contextuales, mejor para polisemia y búsqueda')
print('  • Los embeddings de Word2Vec y BERT están guardados en MongoDB')
print('=' * 65)
